# LearnGuard AI — Stage 1.1: Story Generation and Quality Control

This notebook loads the fine-tuned **KidStory Qwen2.5** model from Google Drive and turns a user topic into one child-friendly story.

### Stage 1 goal

**Input:** topic + target age group  
**Output:** Output: one validated 220–450 word children’s story
**Model:** your fine-tuned merged Qwen2.5 checkpoint

This is the first reusable part of the final pipeline:

`Topic → Story → Answer-span ranking → Question generation → Answer verification → QC gate`

> This notebook performs inference only. It does not retrain or modify your model.

## Before running

In Colab, select:

**Runtime → Change runtime type → T4 GPU → Save**

Then run each cell from top to bottom. A green check beside the GPU test confirms that the correct runtime is active.

In [ ]:
# Confirm that Colab assigned an NVIDIA GPU.
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU was not detected. Select Runtime → Change runtime type → T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())

Sun Jul 26 16:58:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Install inference libraries

In [ ]:
!pip install -q --upgrade \
    "transformers>=4.48,<5" \
    "accelerate>=1.2" \
    "bitsandbytes>=0.45" \
    "safetensors>=0.4" \
    "sentencepiece>=0.2"

print("Libraries installed. If Colab asks you to restart, restart the session and continue.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
Libraries installed. If Colab asks you to restart, restart the session and continue.


## 2. Connect Google Drive and check the model

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

SG_PATH = Path(
    "/content/drive/MyDrive/KidStory-Qwen2.5/final_model_merged"
)
RESULTS_DIR = Path(
    "/content/drive/MyDrive/KidStory-Qwen2.5/learnguard_stage1_results"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required_files = [
    "config.json",
    "tokenizer_config.json",
    "model.safetensors.index.json",
]

if not SG_PATH.exists():
    raise FileNotFoundError(f"Model folder was not found: {SG_PATH}")

missing = [name for name in required_files if not (SG_PATH / name).exists()]
if missing:
    raise FileNotFoundError(f"Required model files are missing: {missing}")

shards = sorted(SG_PATH.glob("model-*.safetensors"))
if not shards:
    raise FileNotFoundError("No model safetensor shards were found.")

print("Model folder:", SG_PATH)
print("Safetensor shards:", len(shards))
print("Stage 1 output folder:", RESULTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model folder: /content/drive/MyDrive/KidStory-Qwen2.5/final_model_merged
Safetensor shards: 4
Stage 1 output folder: /content/drive/MyDrive/KidStory-Qwen2.5/learnguard_stage1_results


## 3. Load the model in 4-bit mode

The merged model is approximately 14.5 GB. Four-bit loading reduces GPU-memory usage so it can run on a Colab T4.

Loading may take several minutes because the weights are read from Google Drive.

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    SG_PATH,
    use_fast=True,
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    SG_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
    quantization_config=quantization_config,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
model.eval()

print("Story-generation model loaded successfully.")
print("Model class:", model.__class__.__name__)
print("Input device:", model.device)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Story-generation model loaded successfully.
Model class: Qwen2ForCausalLM
Input device: cuda:0


## 4. Define the reusable story-generation function

This function will later move into `src/story_generation.py` and be called by the Streamlit application.

In [ ]:
SYSTEM_PROMPT = """You are KidStory, a friendly children's story writer.
Write safe, kind, imaginative and educational stories for children aged 6–14.
Use age-appropriate language, a clear beginning, middle and ending, and a gentle moral.
Avoid horror, graphic violence, romance or sexual content, hate, drugs and alcohol."""


def build_story_prompt(topic: str, age_group: str) -> list[dict]:
    """Create the chat messages used by the fine-tuned Qwen model."""
    clean_topic = topic.strip()
    clean_age = age_group.strip()

    if not clean_topic:
        raise ValueError("Topic cannot be empty.")
    if len(clean_topic) > 150:
        raise ValueError("Please keep the topic below 150 characters.")

    user_prompt = f"""Write ONE child-friendly story about: {clean_topic}

Target age group: {clean_age}
Requirements:
- Include a clear beginning, middle and ending.
- Keep the content safe for children aged 6–14.
- Write 3–6 paragraphs.
- Aim for 220–450 words.
- Preserve the important objects and details from the topic. For example, if the topic says wallet, the story must use a wallet and not replace it with another object.
- End with a gentle one-sentence moral.
- Return only the story. Do not add analysis, labels or bullet points."""

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


@torch.inference_mode()
def generate_story(
    topic: str,
    age_group: str = "8–10",
    max_new_tokens: int = 520,
    temperature: float = 0.8,
    top_p: float = 0.9,
    seed: int | None = 42,
) -> str:
    """Generate one story from a topic using the fine-tuned Qwen model."""
    if seed is not None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    messages = build_story_prompt(topic, age_group)
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=1.08,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    story = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    if not story:
        raise RuntimeError("The model returned an empty story.")

    return story


print("generate_story() is ready.")

generate_story() is ready.


## 5. Enter a topic and generate a story

Change only `TOPIC` and `AGE_GROUP` for your test.

## 6. Run basic output checks

These are engineering checks, not a complete educational-quality evaluation. They help catch empty, extremely short or badly formatted output before the story is sent to the next pipeline stage.

In [ ]:
import re


def check_generated_story(story_text: str) -> dict:
    """Return basic diagnostic information for a generated story."""

    clean_story = story_text.strip()

    words = re.findall(r"\b[\w'-]+\b", clean_story)

    sentences = [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+", clean_story)
        if sentence.strip()
    ]

    return {
        "not_empty": bool(clean_story),
        "word_count": len(words),
        "sentence_count": len(sentences),
        "within_usable_word_range": 220 <= len(words) <= 450,
        "has_clear_ending": len(sentences) >= 5,
    }

def format_story_into_paragraphs(
    story_text: str,
    number_of_paragraphs: int = 3,
) -> str:
    """Divide a story into paragraphs without splitting common titles."""

    clean_story = story_text.strip()

    # Temporarily protect titles that contain full stops.
    protected_story = clean_story

    title_replacements = {
        "Mr.": "Mr<PERIOD>",
        "Mrs.": "Mrs<PERIOD>",
        "Ms.": "Ms<PERIOD>",
        "Dr.": "Dr<PERIOD>",
    }

    for original, replacement in title_replacements.items():
        protected_story = protected_story.replace(
            original,
            replacement,
        )

    sentences = [
        sentence.strip()
        for sentence in re.split(
            r"(?<=[.!?])\s+",
            protected_story,
        )
        if sentence.strip()
    ]

    # Restore the protected full stops.
    sentences = [
        sentence.replace("<PERIOD>", ".")
        for sentence in sentences
    ]

    if len(sentences) < number_of_paragraphs:
        return clean_story

    base_size = len(sentences) // number_of_paragraphs
    remainder = len(sentences) % number_of_paragraphs

    paragraphs = []
    start = 0

    for paragraph_number in range(number_of_paragraphs):
        current_size = base_size

        if paragraph_number < remainder:
            current_size += 1

        end = start + current_size
        paragraphs.append(" ".join(sentences[start:end]))
        start = end

    return "\n\n".join(paragraphs)


def generate_valid_story(
    topic: str,
    age_group: str = "8–10",
    maximum_attempts: int = 3,
) -> tuple[str, dict]:
    """Generate and validate a usable story."""

    best_story = ""
    best_checks = {}

    for attempt in range(1, maximum_attempts + 1):
        print(f"Generation attempt {attempt}/{maximum_attempts}")

        candidate_story = generate_story(
            topic=topic,
            age_group=age_group,
            seed=42 + attempt,
            max_new_tokens=650,
            temperature=0.8,
            top_p=0.9,
        )

        candidate_story = format_story_into_paragraphs(
            candidate_story,
            number_of_paragraphs=3,
        )

        candidate_checks = check_generated_story(candidate_story)

        candidate_checks["generation_attempt"] = attempt
        candidate_checks["seed_used"] = 42 + attempt
        candidate_checks["max_new_tokens"] = 650

        print("Word count:", candidate_checks["word_count"])
        print("Sentences:", candidate_checks["sentence_count"])

        best_story = candidate_story
        best_checks = candidate_checks

        length_ok = candidate_checks["within_usable_word_range"]
        ending_ok = candidate_checks["has_clear_ending"]

        if length_ok and ending_ok:
            print("Story passed the Stage 1 checks.")
            return candidate_story, candidate_checks

        print("Story did not pass. Trying again...\n")

    print(
        "Maximum attempts reached. Returning the final story "
        "for manual review."
    )

    return best_story, best_checks

In [ ]:
@torch.inference_mode()
def revise_story(
    original_story: str,
    topic: str,
    age_group: str = "8–10",
    seed: int = 100,
) -> str:
    """Revise a generated story to improve consistency and coherence."""

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    revision_prompt = f"""
Revise the children's story below.

Original topic:
{topic}

Target age group:
{age_group}

Fix all of the following:

1. Keep every character's name exactly consistent.
2. Make the sequence of events logical.
3. Make ownership of important objects clear.
4. Preserve the original topic and important objects.
5. Remove contradictory or confusing information.
6. Include a clear beginning, problem, resolution and ending.
7. Keep the story between 220 and 450 words.
8. Use 3 readable paragraphs.
9. End with a gentle moral.
10. Return only the revised story.

Original story:
{original_story}
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful children's-story editor. "
                "Correct logical, character and narrative inconsistencies "
                "without changing the main topic."
            ),
        },
        {
            "role": "user",
            "content": revision_prompt,
        },
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=3072,
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=650,
        do_sample=False,
        repetition_penalty=1.08,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_tokens = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    revised_story = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    if not revised_story:
        raise RuntimeError("The model returned an empty revision.")

    return format_story_into_paragraphs(
        revised_story,
        number_of_paragraphs=3,
    )


print("revise_story() is ready.")

revise_story() is ready.


In [ ]:
TOPIC = "honesty when a child finds a lost wallet"
AGE_GROUP = "8–10"

story, checks = generate_valid_story(
    topic=TOPIC,
    age_group=AGE_GROUP,
    maximum_attempts=3,
)

print("\n" + "=" * 80)
print("FINAL FORMATTED STORY")
print("=" * 80)
print(story)

print("\nFINAL CHECKS")

for name, value in checks.items():
    print(f"{name}: {value}")

Generation attempt 1/3
Word count: 245
Sentences: 18
Story passed the Stage 1 checks.

FINAL FORMATTED STORY
Once upon a time, Timmy was walking home from school when he noticed something shiny on the ground. He bent down to pick it up and found a beautiful leather wallet! Timmy looked inside and saw money, credit cards, even an ID card. His heart raced as he thought of all the possibilities - maybe it belonged to someone important? Timmy walked over to Mr. Johnson, their neighbor who always seemed so busy taking care of his garden.

When Mr. John saw what Timmy held in his hand, he smiled warmly. "Timmy," said Mr. Joe, "that looks like quite a valuable item you have there." "It does?" asked Timmy, surprised by how much stuff fit into such a small space. Mr. Joseph explained that people can lose things easily; sometimes they don't know we've picked them up until later. With gratitude, Timms handed back the wallet saying, "I want this person to find out quickly before they miss it!" The

In [ ]:
revised_story = revise_story(
    original_story=story,
    topic=TOPIC,
    age_group=AGE_GROUP,
    seed=100,
)

revised_checks = check_generated_story(revised_story)

print("=" * 80)
print("ORIGINAL STORY")
print("=" * 80)
print(story)

print("\n" + "=" * 80)
print("REVISED STORY")
print("=" * 80)
print(revised_story)

print("\nREVISED STORY CHECKS")

for name, value in revised_checks.items():
    print(f"{name}: {value}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


ORIGINAL STORY
Once upon a time, Timmy was walking home from school when he noticed something shiny on the ground. He bent down to pick it up and found a beautiful leather wallet! Timmy looked inside and saw money, credit cards, even an ID card. His heart raced as he thought of all the possibilities - maybe it belonged to someone important? Timmy walked over to Mr. Johnson, their neighbor who always seemed so busy taking care of his garden.

When Mr. John saw what Timmy held in his hand, he smiled warmly. "Timmy," said Mr. Joe, "that looks like quite a valuable item you have there." "It does?" asked Timmy, surprised by how much stuff fit into such a small space. Mr. Joseph explained that people can lose things easily; sometimes they don't know we've picked them up until later. With gratitude, Timms handed back the wallet saying, "I want this person to find out quickly before they miss it!" They both agreed that finding lost items helps keep everyone happy. And guess what happened next?

In [ ]:
TOPIC_STOPWORDS = {
    "a", "an", "the", "and", "or", "but",
    "when", "where", "who", "why", "how",
    "in", "on", "at", "to", "from", "for",
    "with", "without", "of", "by", "about",
    "is", "are", "was", "were", "be",
    "being", "been", "this", "that",
}


def extract_topic_keywords(topic: str) -> list[str]:
    """Extract useful keywords from the user topic."""

    words = re.findall(
        r"\b[a-zA-Z][a-zA-Z'-]*\b",
        topic.lower(),
    )

    return [
        word
        for word in words
        if word not in TOPIC_STOPWORDS and len(word) > 2
    ]


def check_topic_preservation(
    topic: str,
    story_text: str,
    minimum_coverage: float = 0.40,
) -> dict:
    """Check whether important topic words remain in the story."""

    keywords = extract_topic_keywords(topic)

    story_words = set(
        re.findall(
            r"\b[a-zA-Z][a-zA-Z'-]*\b",
            story_text.lower(),
        )
    )

    present_keywords = [
        keyword
        for keyword in keywords
        if keyword in story_words
    ]

    missing_keywords = [
        keyword
        for keyword in keywords
        if keyword not in story_words
    ]

    if keywords:
        coverage = len(present_keywords) / len(keywords)
    else:
        coverage = 1.0

    # The final meaningful topic keyword is treated as an
    # important object or concept: wallet in the current topic.
    critical_keyword = keywords[-1] if keywords else None

    critical_keyword_preserved = (
        critical_keyword in story_words
        if critical_keyword
        else True
    )

    topic_preserved = (
        coverage >= minimum_coverage
        and critical_keyword_preserved
    )

    return {
        "topic_keywords": keywords,
        "present_topic_keywords": present_keywords,
        "missing_topic_keywords": missing_keywords,
        "topic_keyword_coverage": round(coverage, 3),
        "critical_keyword": critical_keyword,
        "critical_keyword_preserved": critical_keyword_preserved,
        "topic_preserved": topic_preserved,
    }


print("Topic-preservation checker is ready.")

Topic-preservation checker is ready.


In [ ]:
original_topic_check = check_topic_preservation(
    TOPIC,
    story,
)

revised_topic_check = check_topic_preservation(
    TOPIC,
    revised_story,
)

print("ORIGINAL STORY TOPIC CHECK")
for name, value in original_topic_check.items():
    print(f"{name}: {value}")

print("\nREVISED STORY TOPIC CHECK")
for name, value in revised_topic_check.items():
    print(f"{name}: {value}")

ORIGINAL STORY TOPIC CHECK
topic_keywords: ['honesty', 'child', 'finds', 'lost', 'wallet']
present_topic_keywords: ['lost', 'wallet']
missing_topic_keywords: ['honesty', 'child', 'finds']
topic_keyword_coverage: 0.4
critical_keyword: wallet
critical_keyword_preserved: True
topic_preserved: True

REVISED STORY TOPIC CHECK
topic_keywords: ['honesty', 'child', 'finds', 'lost', 'wallet']
present_topic_keywords: ['lost']
missing_topic_keywords: ['honesty', 'child', 'finds', 'wallet']
topic_keyword_coverage: 0.2
critical_keyword: wallet
critical_keyword_preserved: False
topic_preserved: False


In [ ]:
from difflib import SequenceMatcher


def extract_possible_names(story_text: str) -> list[str]:
    """Extract possible character names from a story."""

    candidates = re.findall(
        r"\b[A-Z][a-z]{2,}\b",
        story_text,
    )

    excluded_words = {
        "One", "Once", "The", "When", "While", "With",
        "Without", "After", "Before", "From", "Then",
        "Suddenly", "Seeing", "Feeling", "Every",
        "Everyone", "People", "Inside", "Outside",
        "Just", "That", "This", "There", "They",
        "His", "Her", "Their", "And", "But", "So",
        "Thank", "Target", "Original",
    }

    names = [
        word
        for word in candidates
        if word not in excluded_words
    ]

    return sorted(set(names))


def find_similar_name_variants(
    story_text: str,
    similarity_threshold: float = 0.72,
) -> list[dict]:
    """Find names that may be inconsistent spellings."""

    names = extract_possible_names(story_text)
    possible_variants = []

    for first_index in range(len(names)):
        for second_index in range(first_index + 1, len(names)):
            first_name = names[first_index]
            second_name = names[second_index]

            similarity = SequenceMatcher(
                None,
                first_name.lower(),
                second_name.lower(),
            ).ratio()

            same_beginning = (
                first_name[:3].lower()
                == second_name[:3].lower()
            )

            if (
                first_name != second_name
                and same_beginning
                and similarity >= similarity_threshold
            ):
                possible_variants.append(
                    {
                        "name_1": first_name,
                        "name_2": second_name,
                        "similarity": round(similarity, 3),
                    }
                )

    return possible_variants


def check_name_consistency(story_text: str) -> dict:
    """Return possible character-name inconsistencies."""

    detected_names = extract_possible_names(story_text)

    possible_variants = find_similar_name_variants(
        story_text,
    )

    return {
        "detected_names": detected_names,
        "possible_name_variants": possible_variants,
        "name_consistency_passed": len(possible_variants) == 0,
    }


print("Name-consistency checker is ready.")

Name-consistency checker is ready.


In [ ]:
original_name_check = check_name_consistency(story)

revised_name_check = check_name_consistency(revised_story)

print("ORIGINAL STORY NAME CHECK")
for name, value in original_name_check.items():
    print(f"{name}: {value}")

print("\nREVISED STORY NAME CHECK")
for name, value in revised_name_check.items():
    print(f"{name}: {value}")

ORIGINAL STORY NAME CHECK
detected_names: ['Joe', 'John', 'Johnson', 'Joseph', 'Timmi', 'Timmie', 'Timms', 'Timmy']
possible_name_variants: [{'name_1': 'John', 'name_2': 'Johnson', 'similarity': 0.727}, {'name_1': 'Timmi', 'name_2': 'Timmie', 'similarity': 0.909}, {'name_1': 'Timmi', 'name_2': 'Timms', 'similarity': 0.8}, {'name_1': 'Timmi', 'name_2': 'Timmy', 'similarity': 0.8}, {'name_1': 'Timmie', 'name_2': 'Timms', 'similarity': 0.727}, {'name_1': 'Timmie', 'name_2': 'Timmy', 'similarity': 0.727}, {'name_1': 'Timms', 'name_2': 'Timmy', 'similarity': 0.8}]
name_consistency_passed: False

REVISED STORY NAME CHECK
detected_names: ['Excitedly', 'Green', 'Greene', 'Mrs', 'Sam', 'She', 'Soon', 'Though']
possible_name_variants: [{'name_1': 'Green', 'name_2': 'Greene', 'similarity': 0.909}]
name_consistency_passed: False


In [ ]:
def evaluate_story_quality(
    topic: str,
    story_text: str,
) -> dict:
    """Combine structural, topic and name-consistency checks."""

    structural_checks = check_generated_story(story_text)

    topic_checks = check_topic_preservation(
        topic,
        story_text,
    )

    name_checks = check_name_consistency(story_text)

    rejection_reasons = []
    review_reasons = []

    if not structural_checks["not_empty"]:
        rejection_reasons.append("The story is empty.")

    if not structural_checks["within_usable_word_range"]:
        rejection_reasons.append(
            "The story is outside the 220–450 word range."
        )

    if not structural_checks["has_clear_ending"]:
        rejection_reasons.append(
            "The story may be incomplete or too short."
        )

    if not topic_checks["critical_keyword_preserved"]:
        rejection_reasons.append(
            "The critical topic keyword was not preserved: "
            f"{topic_checks['critical_keyword']}."
        )

    if not topic_checks["topic_preserved"]:
        rejection_reasons.append(
            "The story has insufficient topic-keyword coverage."
        )

    if not name_checks["name_consistency_passed"]:
        review_reasons.append(
            "Possible inconsistent character names were detected."
        )

    if rejection_reasons:
        decision = "REJECT"
    elif review_reasons:
        decision = "REVIEW"
    else:
        decision = "PASS"

    return {
        "decision": decision,
        "rejection_reasons": rejection_reasons,
        "review_reasons": review_reasons,
        "structural_checks": structural_checks,
        "topic_checks": topic_checks,
        "name_checks": name_checks,
    }


print("Combined story-quality gate is ready.")

Combined story-quality gate is ready.


In [ ]:
original_quality = evaluate_story_quality(
    TOPIC,
    story,
)

revised_quality = evaluate_story_quality(
    TOPIC,
    revised_story,
)

print("ORIGINAL STORY DECISION")
print("Decision:", original_quality["decision"])
print(
    "Rejection reasons:",
    original_quality["rejection_reasons"],
)
print(
    "Review reasons:",
    original_quality["review_reasons"],
)

print("\nREVISED STORY DECISION")
print("Decision:", revised_quality["decision"])
print(
    "Rejection reasons:",
    revised_quality["rejection_reasons"],
)
print(
    "Review reasons:",
    revised_quality["review_reasons"],
)

ORIGINAL STORY DECISION
Decision: REVIEW
Rejection reasons: []
Review reasons: ['Possible inconsistent character names were detected.']

REVISED STORY DECISION
Decision: REJECT
Rejection reasons: ['The critical topic keyword was not preserved: wallet.', 'The story has insufficient topic-keyword coverage.']
Review reasons: ['Possible inconsistent character names were detected.']


In [ ]:
def generate_story_with_quality_gate(
    topic: str,
    age_group: str = "8–10",
    maximum_attempts: int = 3,
) -> tuple[str, dict]:
    """
    Generate stories until one passes the automated quality gate.

    If none passes, return the best candidate for manual review.
    """

    candidates = []

    decision_priority = {
        "PASS": 3,
        "REVIEW": 2,
        "REJECT": 1,
    }

    for attempt in range(1, maximum_attempts + 1):
        current_seed = 200 + attempt

        print("=" * 70)
        print(f"QUALITY-GATED GENERATION {attempt}/{maximum_attempts}")
        print("Seed:", current_seed)

        candidate_story = generate_story(
            topic=topic,
            age_group=age_group,
            seed=current_seed,
            max_new_tokens=650,
            temperature=0.65,
            top_p=0.90,
        )

        candidate_story = format_story_into_paragraphs(
            candidate_story,
            number_of_paragraphs=3,
        )

        quality_result = evaluate_story_quality(
            topic,
            candidate_story,
        )

        quality_result["generation_attempt"] = attempt
        quality_result["seed_used"] = current_seed
        quality_result["approved_for_qa"] = (
            quality_result["decision"] == "PASS"
        )

        candidates.append(
            {
                "story": candidate_story,
                "quality": quality_result,
            }
        )

        print("Decision:", quality_result["decision"])
        print(
            "Word count:",
            quality_result["structural_checks"]["word_count"],
        )

        print(
            "Critical keyword preserved:",
            quality_result["topic_checks"][
                "critical_keyword_preserved"
            ],
        )

        print(
            "Possible name variants:",
            quality_result["name_checks"][
                "possible_name_variants"
            ],
        )

        if quality_result["decision"] == "PASS":
            print("Story approved by the automated quality gate.")

            return candidate_story, quality_result

        print("Story was not approved. Trying again...\n")

    best_candidate = max(
        candidates,
        key=lambda item: decision_priority[
            item["quality"]["decision"]
        ],
    )

    print("=" * 70)
    print("No story received PASS.")
    print(
        "Returning the best candidate for manual review:",
        best_candidate["quality"]["decision"],
    )

    best_candidate["quality"]["approved_for_qa"] = False

    return (
        best_candidate["story"],
        best_candidate["quality"],
    )


print("Quality-gated story generator is ready.")

Quality-gated story generator is ready.


In [ ]:
quality_story, quality_report = generate_story_with_quality_gate(
    topic=TOPIC,
    age_group=AGE_GROUP,
    maximum_attempts=3,
)

print("\n" + "=" * 80)
print("SELECTED STORY")
print("=" * 80)
print(quality_story)

print("\nFINAL QUALITY-GATE RESULT")
print("Decision:", quality_report["decision"])
print(
    "Approved for QA:",
    quality_report["approved_for_qa"],
)
print(
    "Generation attempt:",
    quality_report["generation_attempt"],
)
print("Seed used:", quality_report["seed_used"])
print(
    "Rejection reasons:",
    quality_report["rejection_reasons"],
)
print(
    "Review reasons:",
    quality_report["review_reasons"],
)

QUALITY-GATED GENERATION 1/3
Seed: 201
Decision: REJECT
Word count: 206
Critical keyword preserved: True
Possible name variants: [{'name_1': 'Johnson', 'name_2': 'Johnsons', 'similarity': 0.933}, {'name_1': 'Timmy', 'name_2': 'Timmys', 'similarity': 0.909}]
Story was not approved. Trying again...

QUALITY-GATED GENERATION 2/3
Seed: 202
Decision: PASS
Word count: 241
Critical keyword preserved: True
Possible name variants: []
Story approved by the automated quality gate.

SELECTED STORY
It was a sunny day in the park and Emily saw something shiny on the ground. She picked up the object and realized it was a small leather wallet! "Wow," she said to herself, "what could be inside?" She opened the wallet and found some money, a business card, and even an ID tag. There were no clues as to who might have lost it. Suddenly, Emily had an idea - what if she told her friend Timmy about it? He loved learning new things and would love to help find its owner too!

So they went back home together. A

In [ ]:
# Manual review of the automatically selected story

MANUAL_APPROVAL = False

MANUAL_REVIEW_NOTES = [
    "The location becomes unclear after Mr. Johnson calls.",
    "Mrs. Smith appears suddenly without a clear transition.",
    "The relationship between Mrs. Smith and Mr. Johnson needs clarification.",
    "Correct 'make our world better place' to 'make our world a better place'.",
]

final_story_status = (
    "APPROVED"
    if (
        quality_report["approved_for_qa"]
        and MANUAL_APPROVAL
    )
    else "MANUAL_REVISION_REQUIRED"
)

print("Automated decision:", quality_report["decision"])
print("Manual approval:", MANUAL_APPROVAL)
print("Final story status:", final_story_status)

print("\nManual review notes:")

for note_number, note in enumerate(
    MANUAL_REVIEW_NOTES,
    start=1,
):
    print(f"{note_number}. {note}")

Automated decision: PASS
Manual approval: False
Final story status: MANUAL_REVISION_REQUIRED

Manual review notes:
1. The location becomes unclear after Mr. Johnson calls.
2. Mrs. Smith appears suddenly without a clear transition.
3. The relationship between Mrs. Smith and Mr. Johnson needs clarification.
4. Correct 'make our world better place' to 'make our world a better place'.


In [ ]:
reviewed_story = """
It was a sunny day in the park when Emily noticed something shiny on the ground. She picked it up and discovered a small leather wallet. Inside were some money, a business card and an identification card. Emily knew that taking something that belonged to another person would be wrong. She decided to ask her friend Timmy to help her find the owner of the lost wallet.

Emily and Timmy examined the business card and found a telephone number. With help from Emily's mother, they called the number. A worried man named Mr. Johnson answered. He explained that he had lost his wallet while walking through the park earlier that day. Emily's mother arranged for everyone to meet at the nearby community centre, where the wallet could be returned safely.

When Mr. Johnson arrived with his wife, Mrs. Johnson, he correctly described the wallet and everything inside it. Emily then handed it to him. Mr. Johnson thanked Emily and Timmy for protecting his belongings and trying so hard to locate him. Emily felt proud because she had made an honest and responsible decision.

On their way home, Emily and Timmy talked about what had happened. They understood that honesty helps people trust one another and makes the community safer. Emily learned that when a child finds something that belongs to someone else, returning it is more valuable than keeping it. The experience reminded both children that doing the right thing can bring relief and happiness to others.
""".strip()


reviewed_quality = evaluate_story_quality(
    TOPIC,
    reviewed_story,
)

MANUAL_APPROVAL = True

final_story_status = (
    "APPROVED"
    if (
        reviewed_quality["decision"] == "PASS"
        and MANUAL_APPROVAL
    )
    else "NOT_APPROVED"
)

print("=" * 80)
print("HUMAN-REVIEWED STORY")
print("=" * 80)
print(reviewed_story)

print("\nFINAL REVIEW RESULT")
print("Automated decision:", reviewed_quality["decision"])
print("Manual approval:", MANUAL_APPROVAL)
print("Final story status:", final_story_status)
print(
    "Word count:",
    reviewed_quality["structural_checks"]["word_count"],
)
print(
    "Topic coverage:",
    reviewed_quality["topic_checks"]["topic_keyword_coverage"],
)
print(
    "Name variants:",
    reviewed_quality["name_checks"]["possible_name_variants"],
)

HUMAN-REVIEWED STORY
It was a sunny day in the park when Emily noticed something shiny on the ground. She picked it up and discovered a small leather wallet. Inside were some money, a business card and an identification card. Emily knew that taking something that belonged to another person would be wrong. She decided to ask her friend Timmy to help her find the owner of the lost wallet.

Emily and Timmy examined the business card and found a telephone number. With help from Emily's mother, they called the number. A worried man named Mr. Johnson answered. He explained that he had lost his wallet while walking through the park earlier that day. Emily's mother arranged for everyone to meet at the nearby community centre, where the wallet could be returned safely.

When Mr. Johnson arrived with his wife, Mrs. Johnson, he correctly described the wallet and everything inside it. Emily then handed it to him. Mr. Johnson thanked Emily and Timmy for protecting his belongings and trying so hard 

In [ ]:
from datetime import datetime, timezone
import json


final_story_for_stage2 = reviewed_story

timestamp = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S"
)

safe_topic = re.sub(
    r"[^a-z0-9]+",
    "_",
    TOPIC.lower(),
).strip("_")[:50]

quality_record_path = RESULTS_DIR / (
    f"{timestamp}_{safe_topic}_quality_control.json"
)

quality_record = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "model_path": str(SG_PATH),
    "topic": TOPIC,
    "age_group": AGE_GROUP,

    "generation_settings": {
        "selected_attempt": quality_report[
            "generation_attempt"
        ],
        "selected_seed": quality_report["seed_used"],
        "max_new_tokens": 650,
        "temperature": 0.65,
        "top_p": 0.90,
    },

    "stories": {
        "original_generated_story": story,
        "model_revision_attempt": revised_story,
        "quality_gated_generated_story": quality_story,
        "human_reviewed_final_story": reviewed_story,
    },

    "quality_reports": {
        "original_story": original_quality,
        "model_revision": revised_quality,
        "quality_gated_story": quality_report,
        "human_reviewed_story": reviewed_quality,
    },

    "human_review": {
        "manual_approval": MANUAL_APPROVAL,
        "final_status": final_story_status,
        "notes": MANUAL_REVIEW_NOTES,
    },

    "stage2_input": final_story_for_stage2,
}

with quality_record_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        quality_record,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Quality-control record saved:")
print(quality_record_path)

print("\nStage 2 input is ready.")
print(
    "Stage 2 word count:",
    reviewed_quality["structural_checks"]["word_count"],
)

Quality-control record saved:
/content/drive/MyDrive/KidStory-Qwen2.5/learnguard_stage1_results/20260728_044011_honesty_when_a_child_finds_a_lost_wallet_quality_control.json

Stage 2 input is ready.
Stage 2 word count: 247


## 7. Save a reviewed sample

The generated sample is stored as JSON with its generation settings. This provides traceability for testing and portfolio examples.

Review the story manually before adding it to a public GitHub repository.

In [ ]:
from datetime import datetime, timezone
import json
import re

timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
safe_topic = re.sub(r"[^a-z0-9]+", "_", TOPIC.lower()).strip("_")[:50]
result_path = RESULTS_DIR / f"{timestamp}_{safe_topic}.json"

record = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model_path": str(SG_PATH),
    "topic": TOPIC,
    "age_group": AGE_GROUP,
    "generation": {
    "max_new_tokens": checks["max_new_tokens"],
    "temperature": 0.8,
    "top_p": 0.9,
    "seed": checks["seed_used"],
    "attempt": checks["generation_attempt"],
},
    "checks": checks,
    "story": story,
}

with result_path.open("w", encoding="utf-8") as file:
    json.dump(record, file, ensure_ascii=False, indent=2)

print("Saved:", result_path)

Saved: /content/drive/MyDrive/KidStory-Qwen2.5/learnguard_stage1_results/20260726_180705_honesty_when_a_child_finds_a_lost_wallet.json


## Stage 1 completion checklist

Stage 1 is successful when:

- [ ] Colab detects the T4 GPU.
- [ ] Google Drive mounts successfully.
- [ ] All four Qwen safetensor shards are found.
- [ ] The model loads in 4-bit mode without an out-of-memory error.
- [ ] `generate_story()` returns a complete story.
- [ ] The story is saved as JSON.
- [ ] You manually review the story for safety, coherence, age suitability and topic relevance.

### What not to upload to GitHub

Do **not** upload:

- `.safetensors` model weights
- Google Drive credentials
- private tokens
- the entire 2,000-story dataset before confirming publication/IP requirements

### Next stage

After this notebook succeeds, Stage 2 will:

1. pass the generated story to your answer-span ranking function;
2. select suitable candidate answers;
3. load your fine-tuned T5 question-generation model;
4. generate answer-aware comprehension questions.